# ETL Pipeline

From raw data to into the database. 

In [106]:
# Lets create the database and establish the tables:

import polars as pl
import duckdb
from pathlib import Path

dir_path = Path.cwd().parent.parent / "data/refined_raw_data"
database_path = dir_path / "nba_data.duckdb"
con = duckdb.connect(database_path)
con

In [78]:
# Read in the schema we designed    
con.execute(Path(dir_path / "nba_schema.sql").open("r").read())

# Insert Data
Now that we have made our database and established our schema. Lets load in the data

In [79]:
# First thing we will do is player data
raw_data_dir = Path.cwd().parent.parent / "data/raw_data"

In [80]:
# Get heights and weights from another table:
import kagglehub
import shutil 
from pathlib import Path

player_dir = raw_data_dir / "NBA_Player_Data"
player_dir.mkdir(parents=True, exist_ok=True)

if not Path(player_dir / "Players.csv").exists():
    # Download latest version
    path = kagglehub.dataset_download("blitzapurv/nba-players-data-1950-to-2021")
    
    print("Path to dataset files:", path)
    shutil.copytree(path, player_dir, dirs_exist_ok=True)


In [81]:
import sys
from pathlib import Path

# Add the project root to sys.path so 'utils' can be imported
sys.path.append(str(Path.cwd().parent.parent))

from utils.dictionary_mapping import nba_teams

In [82]:
import unicodedata

def remove_accents(name: str) -> str:
    if name is None:
        return None
    return unicodedata.normalize("NFKD", name).encode("ASCII", "ignore").decode("utf-8")

In [83]:
players_df = pl.read_csv(player_dir / "Players.csv")
players_df

,Player,height,weight,collage,born,birth_city,birth_state
i64,str,i64,i64,str,i64,str,str
0,"""Curly Armstrong""",180,77,"""Indiana University""",1918,null,null
1,"""Cliff Barker""",188,83,"""University of Kentucky""",1921,"""Yorktown""","""Indiana"""
2,"""Leo Barnhorst""",193,86,"""University of Notre Dame""",1924,null,null
3,"""Ed Bartels""",196,88,"""North Carolina State Universit…",1925,null,null
4,"""Ralph Beard""",178,79,"""University of Kentucky""",1927,"""Hardinsburg""","""Kentucky"""
…,…,…,…,…,…,…,…
3917,"""Troy Williams""",198,97,"""South Carolina State Universit…",1969,"""Columbia""","""South Carolina"""
3918,"""Kyle Wiltjer""",208,108,"""Gonzaga University""",1992,"""Portland""","""Oregon"""
3919,"""Stephen Zimmerman""",213,108,"""University of Nevada, Las Vega…",1996,"""Hendersonville""","""Tennessee"""


In [84]:
# Clean up Player names
players_df.columns = list(map(str.title, players_df.columns))
players_df = players_df.with_columns(
    pl.col("Player").map_elements(remove_accents, return_dtype=pl.String)
    .str.replace_all(".", "", literal=True)
    .str.replace_all("*", "", literal=True),
    (pl.col("Weight") * 2.20462).round(0).cast(pl.Int64), # Convert to lbs
).drop("Collage", strict=False)
players_df

,Player,Height,Weight,Born,Birth_City,Birth_State
i64,str,i64,i64,i64,str,str
0,"""Curly Armstrong""",180,170,1918,null,null
1,"""Cliff Barker""",188,183,1921,"""Yorktown""","""Indiana"""
2,"""Leo Barnhorst""",193,190,1924,null,null
3,"""Ed Bartels""",196,194,1925,null,null
4,"""Ralph Beard""",178,174,1927,"""Hardinsburg""","""Kentucky"""
…,…,…,…,…,…,…
3917,"""Troy Williams""",198,214,1969,"""Columbia""","""South Carolina"""
3918,"""Kyle Wiltjer""",208,238,1992,"""Portland""","""Oregon"""
3919,"""Stephen Zimmerman""",213,238,1996,"""Hendersonville""","""Tennessee"""


In [85]:
draft_pick_dir = raw_data_dir / "Draft_Pick"
draft_pick_dir.mkdir(parents=True, exist_ok=True)
draft_pick_list = []
for filename in draft_pick_dir.iterdir():
    df = pl.read_csv(filename)
    season = filename.stem.split("_")[-1].split("-")[0]
    df = df.with_columns(
        pl.lit(season).cast(pl.Int64).alias("Draft Class"),
        pl.col("Tm").replace(nba_teams).alias("Draft Team"),
        pl.when(pl.col("College").is_null())
        .then(pl.lit("Unknown"))
        .otherwise(pl.col("College")).alias("College"),
        pl.col("Pk").alias("Pick"),
        pl.col("Player").str.replace_all(".", "", literal=True).str.replace_all("*", "", literal=True).map_elements(remove_accents, return_dtype=pl.String)
    ).drop(["Tm", "Pk"])
    draft_pick_list.append(df)

In [86]:
# Make Master draft pick dataframe
draft_pick_df = pl.concat(draft_pick_list, how="vertical_relaxed")
draft_pick_df

Rk,Player,College,Yrs,G,MP,PTS,TRB,AST,FG%,3P%,FT%,MP_duplicated_0,PTS_duplicated_0,TRB_duplicated_0,AST_duplicated_0,WS,WS/48,BPM,VORP,Draft Class,Draft Team,Pick
i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64
1,"""Anthony Bennett""","""UNLV""","""4""","""151""","""1905""","""658""","""472""","""77""","""0.392""","""0.261""","""0.67""","""12.6""","""4.4""","""3.1""","""0.5""","""0.5""","""0.013""","""-4.8""","""-1.3""",2013,"""Cleveland Cavaliers""",1
2,"""Victor Oladipo""","""Indiana""","""10""","""504""","""16247""","""8503""","""2245""","""1970""","""0.436""","""0.347""","""0.788""","""32.2""","""16.9""","""4.5""","""3.9""","""25.6""","""0.076""","""0.7""","""11.0""",2013,"""Orlando Magic""",2
3,"""Otto Porter Jr""","""Georgetown""","""11""","""527""","""13400""","""5419""","""2599""","""783""","""0.477""","""0.397""","""0.797""","""25.4""","""10.3""","""4.9""","""1.5""","""36.7""","""0.132""","""1.7""","""12.5""",2013,"""Washington Wizards""",3
4,"""Cody Zeller""","""Indiana""","""11""","""552""","""11510""","""4376""","""3125""","""740""","""0.52""","""0.22""","""0.727""","""20.9""","""7.9""","""5.7""","""1.3""","""33.5""","""0.14""","""-0.7""","""3.9""",2013,"""Charlotte Hornets""",4
5,"""Alex Len""","""Maryland""","""12""","""690""","""11754""","""4608""","""3624""","""589""","""0.51""","""0.322""","""0.678""","""17.0""","""6.7""","""5.3""","""0.9""","""23.9""","""0.097""","""-1.9""","""0.3""",2013,"""Phoenix Suns""",5
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
54,"""Cory Hightower""","""Indian Hills Community College""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2000,"""San Antonio Spurs""",54
55,"""Chris Porter""","""Auburn""","""1""","""51""","""1147""","""440""","""189""","""61""","""0.389""","""0.0""","""0.667""","""22.5""","""8.6""","""3.7""","""1.2""","""0.2""","""0.008""","""-4.5""","""-0.7""",2000,"""Golden State Warriors""",55
56,"""Jaquay Walls""","""Colorado""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2000,"""Indiana Pacers""",56


In [87]:
# Merge with player df
master_player_df = players_df.join(draft_pick_df, how="full", left_on="Player", right_on="Player")
master_player_df = master_player_df.with_columns(
    pl.coalesce([
        pl.when((pl.col("Player").is_null()) | (pl.col("Player") == ""))
        .then(None)
        .otherwise(pl.col("Player")),
        pl.when((pl.col("Player_right").is_null()) | (pl.col("Player_right") == ""))
        .then(None)
        .otherwise(pl.col("Player_right")),
        pl.lit("Unknown")
    ]).alias("Player")
).drop(["Player_right"])
master_player_df = master_player_df.filter("Player" != "Unknown")
master_player_df

,Player,Height,Weight,Born,Birth_City,Birth_State,Rk,College,Yrs,G,MP,PTS,TRB,AST,FG%,3P%,FT%,MP_duplicated_0,PTS_duplicated_0,TRB_duplicated_0,AST_duplicated_0,WS,WS/48,BPM,VORP,Draft Class,Draft Team,Pick
i64,str,i64,i64,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,i64
0,"""Curly Armstrong""",180,170,1918,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
1,"""Cliff Barker""",188,183,1921,"""Yorktown""","""Indiana""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2,"""Leo Barnhorst""",193,190,1924,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
3,"""Ed Bartels""",196,194,1925,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
4,"""Ralph Beard""",178,174,1927,"""Hardinsburg""","""Kentucky""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
null,"""Isaiah Jackson""",null,null,null,null,null,22,"""Kentucky""","""4""","""163""","""2438""","""1170""","""697""","""116""","""0.595""","""0.206""","""0.671""","""15.0""","""7.2""","""4.3""","""0.7""","""7.7""","""0.151""","""-0.1""","""1.1""",2021,"""Los Angeles Lakers""",22
null,"""Cameron Johnson""",null,null,null,null,null,11,"""UNC""","""6""","""340""","""9020""","""4395""","""1342""","""664""","""0.453""","""0.392""","""0.851""","""26.5""","""12.9""","""3.9""","""2.0""","""22.9""","""0.122""","""1.6""","""8.2""",2019,"""Minnesota Timberwolves""",11
null,"""Axel Hervelle""",null,null,null,null,null,52,"""Unknown""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2005,"""Denver Nuggets""",52


### Create the helper Function to Map our columns

In [88]:
def insert_and_map(
    con: duckdb.DuckDBPyConnection,
    df: pl.DataFrame,
    table: str,
    key_cols: str | list[str],
    id_col: str | None = None,
    schema: str = "nba_data"
) -> dict | None:
    """
    Inserts unique records into DuckDB, adds manual auto-incrementing ID if needed,
    and returns a key → id mapping (if single key_col and id_col are provided).
    """

    if isinstance(key_cols, str):
        key_cols = [key_cols]

    if id_col is None and len(key_cols) == 1:
        id_col = table.rstrip("s") + "_id"  # fallback

    qualified_table = f'{schema}."{table}"'
    quoted_cols = [f'"{col}"' for col in df.columns]

    # Determine which rows are new (NOT EXISTS)
    con.register("tmp_dataframe", df.unique())
    where_clause = " AND ".join(f't."{col}" = s."{col}"' for col in key_cols)

    new_rows_query = f"""
        SELECT s.*
        FROM tmp_dataframe AS s
        WHERE NOT EXISTS (
            SELECT 1 FROM {qualified_table} AS t
            WHERE {where_clause}
        )
    """
    new_rows = con.sql(new_rows_query).pl()

    # Inject IDs if id_col is provided
    if id_col:
        # Get max ID to start incrementing from
        max_id = con.sql(f"SELECT COALESCE(MAX({id_col}), 0) FROM {qualified_table}").fetchone()[0]
        # Create list of new ids
        new_ids = list(range(max_id + 1, max_id + 1 + new_rows.height))
        # Insert the ids into the new rows 
        new_rows = new_rows.insert_column(0, pl.Series(id_col, new_ids))
        quoted_cols = [f'"{id_col}"'] + quoted_cols

    # Register and insert
    con.register("new_data", new_rows)
    insert_query = f"""
        INSERT INTO {qualified_table} ({', '.join(quoted_cols)})
        SELECT {', '.join(quoted_cols)} FROM new_data;
    """
    con.execute(insert_query)

    # Mapping return (only for single key)
    if id_col and len(key_cols) == 1:
        mapping_df = con.sql(
            f'SELECT "{id_col}", "{key_cols[0]}" FROM {qualified_table}'
        ).pl()
        return dict(zip(mapping_df[key_cols[0]], mapping_df[id_col]))

    return None


### Insert Player Data into the Database

In [89]:
import re
from utils.dictionary_mapping import nba_teams, player_stat_descriptions


def insert_player_data(con, path: Path):
    if path.exists():
        # Load CSVs Player Data
        for csv in path.rglob("*.csv"):
            df = pl.read_csv(csv)
            match = re.search(r"(\d{4})_(\d{4})", csv.stem)
            season = f"{match.group(1)}" if match else None
            df = df.rename({"Age": "Player_Age"})
            stats_columns = df.drop(*["Rk", "Player", "Pos", "Team", "Season", "Awards", "Player-additional"], strict=False).columns
            df = df.with_columns(
                pl.lit(season).cast(pl.Int64).alias("Season"),
                pl.lit(season).cast(pl.Int64).alias("Season"),
                pl.col("Player")
                .str.replace(".", "", literal=True)
                .str.replace("*", "", literal=True)
                .map_elements(remove_accents, return_dtype=pl.String)
            )
            df = df.drop("Player-additional", strict=False)
    
            # Add Team Full Name
            df = df.with_columns(
                pl.col("Team").replace_strict(nba_teams, default="Unknown", return_dtype=pl.String).alias("Team_Full_Name"),
                pl.col("Team").alias("Team_Abbreviation")
            ).drop("Team", strict=False)
            
            # Save DataFrame before College and Award Transformations
            df_before_transformations = df.clone()
            df = df.join(master_player_df, on="Player", how="full")
            df = df.join(master_player_df, on="Player", how="full")
            df = df.drop_nulls(subset=["Season", "Player"])
            # Insert Player Table
            temp_df = pl.DataFrame(
                {
                    "name": df["Player"],
                    "birth_year": df["Born"],
                    "college": df["College"],
                    "birth_state": df["Birth_State"],
                    "birth_city": df["Birth_City"],
                    "draft_pick": df["Pick"],
                    "draft_class": df["Draft Class"],
                    "draft_team": df["Draft Team"]
                    "birth_city": df["Birth_City"],
                    "draft_pick": df["Pick"],
                    "draft_class": df["Draft Class"],
                    "draft_team": df["Draft Team"]
                }
            )
            player_map = insert_and_map(con, temp_df, "players", "name", "player_id")
            # Insert Position Table
            temp_df = pl.DataFrame(
                {
                    "name": df["Pos"],
                }
            )
            position_map = insert_and_map(con, temp_df, "positions", "name", "position_id")
            
            # Insert Player Positions Table
            temp_df = df.with_columns([
                df["Player"].replace(player_map).alias("player_id"),
                df["Pos"].replace(position_map).alias("position_id")
            ]).select(["player_id", "position_id"]).unique()
    
            insert_and_map(con, temp_df, "player_positions", ["player_id", "position_id"])
            
            # Insert team
            temp_df = pl.DataFrame({
                "name": df["Team_Full_Name"],
                "abbreviation": df["Team_Abbreviation"]
            })
            team_map = insert_and_map(con, temp_df, "teams", "name", "team_id")
    
            # Insert seasons
            temp_df = pl.DataFrame({
                "label": df["Season"]
            })
            seasons_map = insert_and_map(con, temp_df, "seasons", "label", "season_id")
            stat_type = path.parts[-1]
            # Stat_types
            temp_df = pl.DataFrame({
                "name": stat_type
            })
            stat_types_map = insert_and_map(con, temp_df, "stat_types", "name", "stat_type_id")
    
            temp_df = pl.DataFrame({
                "name": stats_columns,
                "description": [player_stat_descriptions.get(col, "") for col in stats_columns]
            })
            
            stats_map = insert_and_map(con, temp_df, "stats", "name", "stat_id")
            temp_df = df_before_transformations[*stats_columns, "Player", "Team_Full_Name", "Season"]
            temp_df = temp_df.unpivot(
                index=["Player", "Team_Full_Name", "Season"],
                on=stats_columns,
                variable_name="stat",
                value_name="value"
            )
            temp_df = temp_df.with_columns(
                pl.lit(stat_type).replace(stat_types_map).alias("stat_type_id"),
                pl.col("stat").replace(stats_map).alias("stat_id"),
                pl.col("Player").replace(player_map).alias("player_id"),
                pl.col("Season").replace(seasons_map).alias("season_id"),
                pl.col("Team_Full_Name").replace(team_map).alias("team_id")
            )
            
            temp_df = temp_df.select(["stat_type_id", "stat_id", "player_id", "team_id", "season_id", "value"])
            insert_and_map(con, temp_df, "player_stat_values", ["player_id", "team_id", "season_id", "stat_type_id", "stat_id"], "id")
    
            # Extraction logic using regex
            # Award columns
            temp_df = df.with_columns([
                pl.col("Awards").fill_null("").str.extract(r"MVP-(\d+)", 1).cast(pl.Int64).fill_null(0).alias("MVP"),
                pl.col("Awards").str.extract(r"DPOY-(\d+)", 1).cast(pl.Int64).fill_null(0).alias("DPOY"),
                pl.col("Awards").str.extract(r"ROY-(\d+)", 1).cast(pl.Int64).fill_null(0).alias("ROY"),
                pl.col("Awards").str.extract(r"6MOY-(\d+)", 1).cast(pl.Int64).fill_null(0).alias("6MOY"),
                pl.col("Awards").str.extract(r"CPOY-(\d+)", 1).cast(pl.Int64).fill_null(0).alias("CPOY"),
                pl.col("Awards").str.extract(r"MIP-(\d+)", 1).cast(pl.Int64).fill_null(0).alias("MIP"),
                pl.col("Awards").str.contains("AS").cast(pl.Int64).fill_null(0).alias("All_Star"),
                pl.col("Awards").str.extract(r"DEF(\d)", 1).cast(pl.Int64).fill_null(0).alias("DEF_Team"),
                pl.col("Awards").str.extract(r"NBA(\d)", 1).cast(pl.Int64).fill_null(0).alias("All_NBA")
            ]).drop("Awards")
            
            temp_df = temp_df.select([
                "MVP", "DPOY", "ROY", "6MOY", "CPOY", 
                "MIP", "All_Star", "All_NBA", "DEF_Team", 
                "Team_Full_Name", "Season", "Player"
            ])
            temp_df = temp_df.select([
                pl.col("MVP").alias("mvp_rank"),
                pl.col("DPOY").alias("dpoy_rank"),
                pl.col("ROY").alias("roy_rank"),
                pl.col("6MOY").alias("six_moy_rank"),
                pl.col("CPOY").alias("comeback_player_rank"),
                pl.col("MIP").alias("most_improved"),
                pl.col("All_Star").alias("allstar"),
                pl.col("All_NBA").alias("all_nba_team"),
                pl.col("DEF_Team").alias("all_defense_team"),
                pl.col("Team_Full_Name").replace(team_map).alias("team_id"),
                pl.col("Season").replace(seasons_map).alias("season_id"),
                pl.col("Player").replace(player_map).alias("player_id")
            ])
            insert_and_map(con, temp_df, "award_info", ["player_id", "season_id", "team_id"])
    
            con.commit()
        

In [90]:
for path in player_dir.iterdir():
    if path.is_dir():
        insert_player_data(con, path)

### Insert Team Data into the Database

In [91]:
from utils.dictionary_mapping import team_stat_descriptions
from utils.dictionary_mapping import team_stat_descriptions

def insert_team_data(con, path: Path):
    if path.exists():
        for csv in path.rglob("*.csv"):
            stat_type = path.parts[-1]
            match stat_type:
                case "Advanced_Data":
                    df = pl.read_csv(csv, skip_rows=1, truncate_ragged_lines=True)
                    df = df.drop(["_duplicated_0", "_duplicated_1", ""])
                    df = df.rename({
                        "eFG%": "Team_eFG%",
                        "TOV%": "Team_TOV%",
                        "ORB%": "Team_ORB%",
                        "FT/FGA": "Team_FT/FGA",
                        "eFG%_duplicated_0": "Opponent_eFG%",
                        "TOV%_duplicated_0": "Opponent_TOV%",
                        "DRB%": "Opponent_DRB%",
                        "FT/FGA_duplicated_0": "Opponent_FT/FGA",
                        "Rk": "Rank",
                        "W": "Win",
                        "L": "Lose",
                        "Attend.": "Total Attendance",
                        "Attend./G": "Avg_Home_Game_Attendance"
                    })
                case _:
                    df = pl.read_csv(csv)
            name = csv.stem.split("_")[-1].split("-")[0]
            df = df.with_columns(
                pl.lit(name).alias("Season"),
                pl.col("Team").str.contains("*", literal=True).alias("Playoffs"),
                pl.col("Team").str.replace_all("*", "", literal=True),
            )
            # Don't need and can agg ourselves
            df = df.filter(pl.col("Team") != "League Average") 
            
            # Insert team
            reversed_nba_teams = {v: k for k, v in nba_teams.items()}
            temp_df = pl.DataFrame({
                "name": df["Team"],
                "abbreviation": df["Team"].replace_strict(reversed_nba_teams, default="Unknown", return_dtype=pl.String)
            })
            team_map = insert_and_map(con, temp_df, "teams", "name", "team_id")
            
            # Insert seasons
            temp_df = pl.DataFrame({
                "label": df["Season"]
            })
            seasons_map = insert_and_map(con, temp_df, "seasons", "label", "season_id")
            
            team_season_info = ["Arena", "Total Attendance", "Avg_Home_Game_Attendance", "Playoffs"]
            temp_df = df.drop(team_season_info + ["Season", "Team"], strict=False)
            
            # Insert into Stat Columns
            stat_columns = temp_df.columns
            temp_df = pl.DataFrame({
                "name": stat_columns,
                "description": [team_stat_descriptions.get(col, "") for col in stat_columns]
            })
            stats_map = insert_and_map(con, temp_df, "stats", "name", "stat_id")
            
            stat_type = path.parts[-1]
            # Stat_types
            temp_df = pl.DataFrame({
                "name": stat_type
            })
            stat_types_map = insert_and_map(con, temp_df, "stat_types", "name", "stat_type_id")

            temp_df = df.select(stat_columns + ["Team", "Season"])
            temp_df = temp_df.unpivot(
                index=["Team", "Season"],
                on=stat_columns,
                variable_name="stat",
                value_name="value"
            )
            temp_df = temp_df.with_columns(
                pl.lit(stat_type).replace(stat_types_map).alias("stat_type_id"),
                pl.col("stat").replace(stats_map).alias("stat_id"),
                pl.col("Season").replace(seasons_map).alias("season_id"),
                pl.col("Team").replace(team_map).alias("team_id")
            )

            temp_df = temp_df.select(["stat_id", "team_id", "season_id", "value", "stat_type_id"])
            insert_and_map(con, temp_df, "team_stat_values", ["team_id", "season_id", "stat_id", "stat_type_id"], "id")
            if set(team_season_info).issubset(df.columns):
                # If the team season info is present, we can insert it
                temp_df = df.select(team_season_info + ["Team", "Season"])
                temp_df = temp_df.with_columns(
                    pl.col("Season").replace(seasons_map).alias("season_id"),
                    pl.col("Team").replace(team_map).alias("team_id"),
                    pl.col("Arena").alias("arena"),
                    pl.col("Total Attendance").alias("attendance"),
                    pl.col("Avg_Home_Game_Attendance").alias("attendance_per_game"),
                    pl.col("Playoffs").alias("made_playoffs")
                )
                temp_df = temp_df.select(["team_id", "season_id", "arena", "attendance", "attendance_per_game", "made_playoffs"])
                insert_and_map(con, temp_df, "team_season_info", ["team_id", "season_id"])
            
            con.commit()


In [92]:
team_dir = raw_data_dir / "Team_Stats"

for path in team_dir.iterdir():
    if path.is_dir():
        insert_team_data(con, path)

### Insert Draft Combine Data

In [93]:
from utils.dictionary_mapping import player_combine_description
from utils.dictionary_mapping import player_combine_description
import pandas as pd
import logging
def insert_draft_combine_data(con, path: Path) -> None:
    if path.exists():
        try:
            df = pl.read_csv(path, null_values="NA", ignore_errors=True)
        except pl.exceptions.ComputeError as e:
            logging.error("Trying to load this manually...")
            player_list = []
            table_list = []
            with path.open("r", newline='\n', encoding="utf-8") as f:
                reader = csv.reader(f)  # handles commas and quotes correctly
                header = next(reader)
                new_header = [header[0]] + header[2:]
            
                for row in reader:
                    if len(row) < 4:
                        continue  # skip malformed rows
                    name = row[1].split(",")
                    name = " ".join(name).strip("\n")
                    player_list.append(name)
                    cleaned_row = [row[0]] + row[2:]  
                    # remove name parts from original row
                    table_list.append(cleaned_row)
    
                # Create dictionary
                data_dict = {col: [row[i] for row in table_list] for i, col in enumerate(new_header)}
                data_dict["Player"] = player_list
        
                # Create DataFrame
                df = pl.DataFrame(data_dict, nan_to_null=True)
        
        df.columns = list(map(str.title, df.columns))
        df = df.with_columns(
            pl.col("Player")
            .str.replace_all(".", "", literal=True)
            .str.replace_all("*", "", literal=True)
            .str.split_exact(", ", 1)
            .struct.rename_fields(["last", "first"])
            .map_elements(lambda s: f"{s['first']} {s['last']}", return_dtype=pl.String)
            .map_elements(remove_accents, return_dtype=pl.String)
        )
        df = df.rename({
            "Pos": "Position",
            "Hgt": "Height",
            "Wgt": "Weight",
            "Bf": "Body_Fat",
            "Wngspn": "Wingspan",
            "Stndrch": "Standing_Reach",
            "Handl": "Length_of_Hand",
            "Handw": "Width_of_Hand",
            'Stndvert': "Standing_Vertical",
             'Lpvert': "Max_Vertical_Leap",
             'Lane': "Lane_Agility_Time",
             'Shuttle': "Shuttle_Run",
             'Sprint': "3_Quarter_Sprint",
             'Bench': "Max_Bench_Press",
             'Bar': "Ration_of_Wingspan",
             'Pan': "Length_x_Width_Hand",
             'Pbhgt': "Standing_Reach_Vertical",
             'Pdhgt': "Standing_Reach_Max_Vertical"
        })
        df = df.with_columns(
            pl.when(pl.col("Ration_of_Wingspan").str.strip_chars().eq("NA"))
            .then(None)
            .otherwise(pl.col("Ration_of_Wingspan"))
            .str.strip_suffix("%")
            .cast(pl.Float64).truediv(100)
            .alias("Ration_of_Wingspan"),
        
            pl.col("Year")
            .cast(pl.Int64)
            .map_elements(lambda x: f"{x}", return_dtype=pl.String).cast(pl.Int64)
            .map_elements(lambda x: f"{x}", return_dtype=pl.String).cast(pl.Int64)
            .alias("Season")
        ).drop("Year")
        # Insert Player Data (Don't have some features so we need to query the DB)
        # Get unique player names
        player_names = df["Player"].unique().to_list()
        
        # Create placeholders for a safe query
        placeholders = ",".join(["?"] * len(player_names))
    
        # Execute query directly using duckdb, then convert to polars DataFrame
        query = f"SELECT name, birth_year, college, player_id FROM nba_data.players WHERE name IN ({placeholders})"
        arrow_table = con.execute(query, player_names).arrow()
        existing_players_df = pl.from_arrow(arrow_table)
        # Merge the existing mappings with your dataset
        players_df = df.select("Player").join(existing_players_df, left_on="Player", right_on="name", how="left")
        
        temp_df = pl.DataFrame({
            "name": players_df["Player"],
            "birth_year": players_df["birth_year"],
            "college": players_df["college"],
            "birth_state": None,
            "birth_city": None
        })
        player_map = insert_and_map(con, temp_df, "players", "name", "player_id")
        # Save Position and Map them
        player_positions = df.with_columns(
            pl.col("Position").str.split("-").alias("Position_List")
        ).explode("Position_List").drop_nulls()
        
        player_positions = player_positions.filter(pl.col("Position_List") != "NA")
        # Insert Position Table
        temp_df = pl.DataFrame(
            {
                "name": player_positions["Position_List"],
            }
        )
        position_map = insert_and_map(con, temp_df, "positions", "name", "position_id")
        
        # Insert Player Positions Table
        temp_df = player_positions.with_columns([
            player_positions["Player"].replace(player_map).alias("player_id"),
            player_positions["Position_List"].replace(position_map).alias("position_id")
        ]).select(["player_id", "position_id"]).unique()
        insert_and_map(con, temp_df, "player_positions", ["player_id", "position_id"])
        
        # Insert seasons
        temp_df = pl.DataFrame({
            "label": df["Season"]
        })
        seasons_map = insert_and_map(con, temp_df, "seasons", "label", "season_id")
        
        # Insert into Stat Columns
        temp_df = df.drop(["Season", "Player", "Position"])
        stat_columns = list(temp_df.columns)
        temp_df = pl.DataFrame({
            "name": stat_columns,
            "description": [player_combine_description.get(col, "") for col in stat_columns]
        })
        stats_map = insert_and_map(con, temp_df, "stats", "name", "stat_id")
        
        temp_df = df.select(stat_columns + ["Player", "Season"])
        temp_df = temp_df.unpivot(
            index=["Player", "Season"],
            on=stat_columns,
            variable_name="stat",
            value_name="value"
        )
        temp_df = temp_df.with_columns(
            pl.col("stat").replace(stats_map).alias("stat_id"),
            pl.col("Season").replace(seasons_map).alias("season_id"),
            pl.col("Player").replace(player_map).alias("player_id"),
            pl.when(pl.col("value") == 'NA')
            .then(None)
            .otherwise(pl.col("value")).alias("value")
        )
        
        temp_df = temp_df.select(["stat_id", "player_id", "season_id", "value"])
        insert_and_map(con, temp_df, "combine_stat_values", ["player_id", "season_id", "stat_id"], "id")
        
        con.commit()


In [94]:
import kagglehub
import shutil 
from pathlib import Path
import csv

draft_dir = raw_data_dir / "NBA_Draft_Combine"
draft_dir.mkdir(parents=True, exist_ok=True)

if not Path(draft_dir / "Draft Combine - Kaggle.csv").exists():
    # Download latest version
    path = kagglehub.dataset_download("marcusfern/nba-draft-combine")
    
    print("Path to dataset files:", path)
    shutil.copytree(path, draft_dir, dirs_exist_ok=True)
for path in draft_dir.iterdir():
    if path.suffix == ".csv":
        insert_draft_combine_data(con, path)

In [95]:
from utils.dictionary_mapping import player_combine_description
from utils.dictionary_mapping import player_combine_description

def insert_mvp_data(con, path: Path):
    if path.exists():
        for csv in path.rglob("*.csv"):
            # Schema Definition
            schema = {
                'Rank': pl.String, 'Player': pl.String, 'Age': pl.Int64, 'Tm': pl.String, 'First': pl.Int64, 'Pts Won': pl.Int64, 'Pts Max': pl.Int64, 'Share': pl.Float64, 'G': pl.Int64, 'MP': pl.Float64, 'PTS': pl.Float64, 'TRB': pl.Float64, 'AST': pl.Float64, 'STL': pl.Float64, 'BLK': pl.Float64, 'FG%': pl.Float64, '3P%': pl.Float64, 'FT%': pl.Float64, 'WS': pl.Float64, 'WS/48': pl.Float64, '-9999': pl.String}
    
            df = pl.read_csv(csv, schema=schema)
            df = df.drop(pl.col("-9999"), strict=False)
            
            # Get season from file name. Trying to be creative
            years = csv.stem.split("_")[-2]         
            season = f"{years}"
            df = df.rename({"Tm": "Team_Abbreviation"})

            season = f"{years}"
            df = df.rename({"Tm": "Team_Abbreviation"})

            df = df.with_columns(
                pl.col("Rank").str.replace("T", "").cast(pl.Int64),
                pl.col("Team_Abbreviation").map_elements(lambda x: nba_teams.get(x, "Unknown"), return_dtype=pl.String).alias("Team_Full_Name"),
                pl.lit(season).cast(pl.Int64).alias("Season"),
                pl.lit(season).cast(pl.Int64).alias("Season"),
                pl.col("Player")
                .str.replace_all(".", "", literal=True)
                .str.replace_all("*", "", literal=True)
                .map_elements(remove_accents, return_dtype=pl.String)
            )


            # Time to do mapping and insertion
            
            # Insert Player Data (Don't have some features so we need to query the DB)
            # Get unique player names
            player_names = df["Player"].unique().to_list()
            # Create placeholders for a safe query
            placeholders = ",".join(["?"] * len(player_names))
        
            # Execute query directly using duckdb, then convert to polars DataFrame
            query = f"SELECT * FROM nba_data.players WHERE name IN ({placeholders})"
            query = f"SELECT * FROM nba_data.players WHERE name IN ({placeholders})"
            arrow_table = con.execute(query, player_names).arrow()
            existing_players_df = pl.from_arrow(arrow_table)
            # Merge the existing mappings with your dataset
            players_df = df.select("Player").join(existing_players_df, left_on="Player", right_on="name", how="left")

            temp_df = players_df.drop("player_id").rename({"Player": "name"}).clone()

            temp_df = players_df.drop("player_id").rename({"Player": "name"}).clone()
            player_map = insert_and_map(con, temp_df, "players", "name", "player_id")
            
            # Insert seasons
            temp_df = pl.DataFrame({
                "label": df["Season"]
            })
            seasons_map = insert_and_map(con, temp_df, "seasons", "label", "season_id")
            
            # Insert team
            temp_df = pl.DataFrame({
                "name": df["Team_Full_Name"],
                "abbreviation": df["Team_Abbreviation"]
            })
            team_map = insert_and_map(con, temp_df, "teams", "name", "team_id")
            
            # Insert into MVP Voting
            temp_df = df.with_columns(
                pl.col("Player").replace(player_map).alias("player_id"),
                pl.col("Team_Full_Name").replace(team_map).alias("team_id"),
                pl.col("Season").replace(seasons_map).alias("season_id"),
                pl.col("Rank").alias("rank"),
                pl.col("First").alias("first_place_votes"),
                pl.col("Pts Won").alias("points_won"),
                pl.col("Age").alias("age"),
                pl.col("Pts Max").alias("points_max"),
                pl.col("Share").alias("point_share")
            )
            
            temp_df = temp_df["player_id", "team_id", "season_id", "rank", "first_place_votes", "points_won", "age", "points_max", "point_share"]
            
            insert_and_map(con, temp_df, "mvp_voting", ["player_id", "team_id", "season_id"], "vote_id")
            
            con.commit()

In [96]:
mvp_dir = raw_data_dir / "MVP_Voting"

for path in mvp_dir.iterdir():
    if path.is_dir():
        insert_mvp_data(con, path)

In [97]:
def insert_roy_data(con, path: Path):
    if path.exists():
        for csv in path.rglob("*.csv"):

            df = pl.read_csv(csv)
            df = df.drop(pl.col("-9999"), strict=False)
            
            # Get season from file name. Trying to be creative
            years = csv.stem.split("_")[-2]         
            season = f"{years}"

            df = df.rename({
                "Tm": "Team_Abbreviation", 
            })
            df = df.with_columns(
                pl.col("Rank").cast(pl.String).str.replace("T", "", literal=True).cast(pl.Int64).alias("Rank"),
                pl.col("Team_Abbreviation").map_elements(lambda x: nba_teams.get(x, "Unknown"), return_dtype=pl.String).alias("Team_Full_Name"),
                pl.lit(season).cast(pl.Int64).alias("Season"),
                pl.col("Player")
                .str.replace_all(".", "", literal=True)
                .str.replace_all("*", "", literal=True)
                .map_elements(remove_accents, return_dtype=pl.String).alias("Player")
            )
            # Time to do mapping and insertion

            # Insert Player Data (Don't have some features so we need to query the DB)
            # Get unique player names
            player_names = df["Player"].unique().to_list()
            # Create placeholders for a safe query
            placeholders = ",".join(["?"] * len(player_names))

            # Execute query directly using duckdb, then convert to polars DataFrame
            query = f"SELECT * FROM nba_data.players WHERE name IN ({placeholders})"
            arrow_table = con.execute(query, player_names).arrow()
            existing_players_df = pl.from_arrow(arrow_table)
            # Merge the existing mappings with your dataset
            players_df = df.select("Player").join(existing_players_df, left_on="Player", right_on="name", how="left")

            temp_df = players_df.drop("player_id").rename({"Player": "name"}).clone()
            player_map = insert_and_map(con, temp_df, "players", "name", "player_id")

            # Insert seasons
            temp_df = pl.DataFrame({
                "label": df["Season"]
            })
            seasons_map = insert_and_map(con, temp_df, "seasons", "label", "season_id")

            # Insert team
            temp_df = pl.DataFrame({
                "name": df["Team_Full_Name"],
                "abbreviation": df["Team_Abbreviation"]
            })
            team_map = insert_and_map(con, temp_df, "teams", "name", "team_id")

            # Insert into ROY Voting
            temp_df = df.with_columns(
                pl.col("Player").replace(player_map).alias("player_id"),
                pl.col("Team_Full_Name").replace(team_map).alias("team_id"),
                pl.col("Season").replace(seasons_map).alias("season_id"),
                pl.col("Rank").alias("rank"),
                pl.col("First").alias("first_place_votes"),
                pl.col("Pts Won").alias("points_won"),
                pl.col("Age").alias("age"),
                pl.col("Pts Max").alias("points_max"),
                pl.col("Share").alias("point_share")
            )

            temp_df = temp_df["player_id", "team_id", "season_id", "rank", "first_place_votes", "points_won", "age", "points_max", "point_share"]

            insert_and_map(con, temp_df, "roy_voting", ["player_id", "team_id", "season_id"], "vote_id")

            con.commit()

In [98]:
roy_dir = raw_data_dir / "Rookie_OTY"

for path in roy_dir.iterdir():
    if path.is_dir():
        insert_roy_data(con, path)

In [99]:
def insert_all_nba_data(con, path: Path):
    if path.exists():
        for csv in path.rglob("*.csv"):
            # Schema Definition

            df = pl.read_csv(csv)
            df = df.drop(pl.col("-9999"), strict=False)
            
            # Get season from file name. Trying to be creative
            years = csv.stem.split("_")[-1].split("-")[0]
            season = f"{years}"

            df = df.rename({
                "Tm": "Team_Abbreviation", 
                "# Tm": "All NBA Team",
                "1st Tm": "First",
                "2nd Tm": "Second",
                "3rd Tm": "Third"
            })
            
            df = df.with_columns(
                pl.col("Team_Abbreviation").map_elements(lambda x: nba_teams.get(x, "Unknown"), return_dtype=pl.String).alias("Team_Full_Name"),
                pl.lit(season).cast(pl.Int64).alias("Season"),
                pl.col("Player")
                .str.replace_all(".", "", literal=True)
                .str.replace_all("*", "", literal=True)
                .map_elements(remove_accents, return_dtype=pl.String).alias("Player")
            )
            # Time to do mapping and insertion

            # Insert Player Data (Don't have some features so we need to query the DB)
            # Get unique player names
            player_names = df["Player"].unique().to_list()
            # Create placeholders for a safe query
            placeholders = ",".join(["?"] * len(player_names))

            # Execute query directly using duckdb, then convert to polars DataFrame
            query = f"SELECT * FROM nba_data.players WHERE name IN ({placeholders})"
            arrow_table = con.execute(query, player_names).arrow()
            existing_players_df = pl.from_arrow(arrow_table)
            # Merge the existing mappings with your dataset
            players_df = df.select("Player").join(existing_players_df, left_on="Player", right_on="name", how="left")

            temp_df = players_df.drop("player_id").rename({"Player": "name"}).clone()
            player_map = insert_and_map(con, temp_df, "players", "name", "player_id")

            # Insert seasons
            temp_df = pl.DataFrame({
                "label": df["Season"]
            })
            seasons_map = insert_and_map(con, temp_df, "seasons", "label", "season_id")

            # Insert team
            temp_df = pl.DataFrame({
                "name": df["Team_Full_Name"],
                "abbreviation": df["Team_Abbreviation"]
            })
            team_map = insert_and_map(con, temp_df, "teams", "name", "team_id")

            df = df.with_columns(
                pl.when((pl.col("All NBA Team") == "1st") | (pl.col("All NBA Team") == "1T"))
                .then(pl.lit("1ST Team"))
                .when((pl.col("All NBA Team") == "2nd") | (pl.col("All NBA Team") == "2T"))
                .then(pl.lit("2ND Team"))
                .when((pl.col("All NBA Team") == "3rd") | (pl.col("All NBA Team") == "3T"))
                .then(pl.lit("3RD Team"))
                .when(pl.col("All NBA Team") == "ORV")
                .then(pl.lit("Received Votes"))
                .otherwise(pl.lit("No Votes")).alias("All NBA Team")
            )
            # Insert into All NBA Voting
            temp_df = df.with_columns(
                pl.col("Player").replace(player_map).alias("player_id"),
                pl.col("Team_Full_Name").replace(team_map).alias("team_id"),
                pl.col("Season").replace(seasons_map).alias("season_id"),
                pl.col("All NBA Team").alias("rank"),
                pl.col("First").alias("first_place_votes"),
                pl.col("Second").alias("second_place_votes"),
                pl.col("Third").alias("third_place_votes"),
                pl.col("Pts Won").alias("points_won"),
                pl.col("Age").alias("age"),
                pl.col("Pts Max").alias("points_max"),
                pl.col("Share").alias("points_share")
            )
            temp_df = temp_df["player_id", "team_id", "season_id", "rank", "first_place_votes", "second_place_votes", "third_place_votes", "points_won", "age", "points_max", "points_share"]

            insert_and_map(con, temp_df, "all_nba_voting", ["player_id", "team_id", "season_id"], "vote_id")
            
            con.commit()

In [100]:
all_nba_dir = raw_data_dir / "All_Nba_Data"

for path in all_nba_dir.iterdir():
    if path.is_dir():
        insert_all_nba_data(con, path)

In [101]:
def insert_all_rookie_data(con, path: Path):
    if path.exists():
        for csv in path.rglob("*.csv"):
            # Schema Definition

            df = pl.read_csv(csv)
            df = df.drop(pl.col("-9999"), strict=False)
            
            # Get season from file name. Trying to be creative
            years = csv.stem.split("_")[-1].split("-")[0]
            season = f"{years}"

            df = df.rename({
                "Tm": "Team_Abbreviation", 
                "# Tm": "All Rookie Team",
                "1st Tm": "First",
                "2nd Tm": "Second",
            })
            
            df = df.with_columns(
                pl.col("Team_Abbreviation").map_elements(lambda x: nba_teams.get(x, "Unknown"), return_dtype=pl.String).alias("Team_Full_Name"),
                pl.lit(season).cast(pl.Int64).alias("Season"),
                pl.col("Player")
                .str.replace_all(".", "", literal=True)
                .str.replace_all("*", "", literal=True)
                .map_elements(remove_accents, return_dtype=pl.String).alias("Player")
            )
            # Time to do mapping and insertion

            # Insert Player Data (Don't have some features so we need to query the DB)
            # Get unique player names
            player_names = df["Player"].unique().to_list()
            # Create placeholders for a safe query
            placeholders = ",".join(["?"] * len(player_names))

            # Execute query directly using duckdb, then convert to polars DataFrame
            query = f"SELECT * FROM nba_data.players WHERE name IN ({placeholders})"
            arrow_table = con.execute(query, player_names).arrow()
            existing_players_df = pl.from_arrow(arrow_table)
            # Merge the existing mappings with your dataset
            players_df = df.select("Player").join(existing_players_df, left_on="Player", right_on="name", how="left")

            temp_df = players_df.drop("player_id").rename({"Player": "name"}).clone()
            player_map = insert_and_map(con, temp_df, "players", "name", "player_id")

            # Insert seasons
            temp_df = pl.DataFrame({
                "label": df["Season"]
            })
            seasons_map = insert_and_map(con, temp_df, "seasons", "label", "season_id")

            # Insert team
            temp_df = pl.DataFrame({
                "name": df["Team_Full_Name"],
                "abbreviation": df["Team_Abbreviation"]
            })
            team_map = insert_and_map(con, temp_df, "teams", "name", "team_id")

            df = df.with_columns(
                pl.when((pl.col("All Rookie Team") == "1st") | (pl.col("All Rookie Team") == "1T"))
                .then(pl.lit("1ST Team"))
                .when((pl.col("All Rookie Team") == "2nd") | (pl.col("All Rookie Team") == "2T"))
                .then(pl.lit("2ND Team"))
                .when(pl.col("All Rookie Team") == "ORV")
                .then(pl.lit("Received Votes"))
                .otherwise(pl.lit("No Votes")).alias("All Rookie Team")
            )
            # Insert into All NBA Voting
            temp_df = df.with_columns(
                pl.col("Player").replace(player_map).alias("player_id"),
                pl.col("Team_Full_Name").replace(team_map).alias("team_id"),
                pl.col("Season").replace(seasons_map).alias("season_id"),
                pl.col("All Rookie Team").alias("rank"),
                pl.col("First").alias("first_place_votes"),
                pl.col("Second").alias("second_place_votes"),
                pl.col("Pts Won").alias("points_won"),
                pl.col("Age").alias("age"),
                pl.col("Pts Max").alias("points_max"),
                pl.col("Share").alias("points_share")
            )
            temp_df = temp_df["player_id", "team_id", "season_id", "rank", "first_place_votes", "second_place_votes", "points_won", "age", "points_max", "points_share"]

            insert_and_map(con, temp_df, "all_rookie_voting", ["player_id", "team_id", "season_id"], "vote_id")
            
            con.commit()

In [102]:
all_rookie_dir = raw_data_dir / "All_Rookie_Team"

for path in all_rookie_dir.iterdir():
    if path.is_dir():
        insert_all_rookie_data(con, path)

In [107]:
con.close()